# Conda Environment: 02_Community-plotting

---

# Library Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

---

# Creating Output Directories

In [2]:
!mkdir -p ../Figures/Community
!mkdir -p ../Datasets/Relative_abundance

---

# Setting Up Data For Plotting

## Import Sample Metadata

In [4]:
sample_data = pd.read_csv("../Datasets/sample-data.csv")

## Import Taxonomy Data

In [3]:
# Taxonomy table
taxonomy = pd.read_csv("../Preprocessing/asvcleaner/cleaned/cleaned_tax_table.tsv", sep = "\t")
taxonomy.index = taxonomy["ASV_ID"]
taxonomy = taxonomy.drop(["ASV_ID", "Species", "confidence", "sequence"], axis=1)

# Species table (produced by dada2 addSpecies function)
species = pd.read_csv("../Preprocessing/asvcleaner/cleaned/cleaned_species_table.tsv", sep = "\t")
species.index = species["ASV_ID"]
species = species["Species_exact"]

# Merge
taxonomy = pd.merge(left= taxonomy, right= species, how= 'left', left_index= True, right_index= True)
taxonomy = taxonomy.reset_index()
taxonomy = taxonomy.fillna('Unassigned')

## Import Sequence Table Subsets

In [5]:
s1_active = pd.read_csv("../Datasets/SequenceTables/StageOneActive.csv", index_col= 0)
s2_active = pd.read_csv("../Datasets/SequenceTables/StageTwoActive.csv", index_col= 0)
s1_resident_w8 = pd.read_csv("../Datasets/SequenceTables/StageOneResidentWeekEight.csv", index_col= 0)
s2_resident_w8 = pd.read_csv("../Datasets/SequenceTables/StageTwoResidentWeekEight.csv", index_col= 0)
s2_resident = pd.read_csv("../Datasets/SequenceTables/StageTwoResident.csv", index_col= 0)

## Convert Sequence Tables To Relative Abundance

In [6]:
s1_active = s1_active / s1_active.sum(axis= 0)
s2_active = s2_active / s2_active.sum(axis= 0)
s1_resident_w8 = s1_resident_w8 / s1_resident_w8.sum(axis= 0)
s2_resident_w8 = s2_resident_w8 / s2_resident_w8.sum(axis= 0)
s2_resident = s2_resident / s2_resident.sum(axis= 0)

## Remove Rows (ASVs) With No Counts From Sequence Tables

In [8]:
def remove_zeroes(seqtable):
    # create empty list to populate with ASVs to keep
    keep = []

    # find rows with 0 counts
    for row in seqtable.itertuples():
        if seqtable.loc[row[0]].sum() > 0:
            keep.append(row[0])

    # subset data to rows with non-zero counts
    output = seqtable.loc[keep]
    
    return output

In [9]:
# Remove ASVs with 0 counts from each subset
s1_active = remove_zeroes(s1_active)
s2_active = remove_zeroes(s2_active)
s1_resident_w8 = remove_zeroes(s1_resident_w8)
s2_resident_w8 = remove_zeroes(s2_resident_w8)
s2_resident = remove_zeroes(s2_resident)

# Export to files
s1_active.to_csv("../Datasets/Relative_abundance/StageOneActive_RA.csv", index= True)
s2_active.to_csv("../Datasets/Relative_abundance/StageTwoActive_RA.csv", index= True)
s1_resident_w8.to_csv("../Datasets/Relative_abundance/StageOneResidentWeekEight_RA.csv", index= True)
s2_resident_w8.to_csv("../Datasets/Relative_abundance/StageTwoResidentWeekEight_RA.csv", index= True)
s2_resident.to_csv("../Datasets/Relative_abundance/StageTwoResident_RA.csv", index= True)


# Melt to long format
s1_active = pd.melt(s1_active, ignore_index= False, var_name= 'Sample', value_name= 'Count').reset_index()
s2_active = pd.melt(s2_active, ignore_index= False, var_name= 'Sample', value_name= 'Count').reset_index()
s1_resident_w8 = pd.melt(s1_resident_w8, ignore_index= False, var_name= 'Sample', value_name= 'Count').reset_index()
s2_resident_w8 = pd.melt(s2_resident_w8, ignore_index= False, var_name= 'Sample', value_name= 'Count').reset_index()
s2_resident = pd.melt(s2_resident, ignore_index= False, var_name= 'Sample', value_name= 'Count').reset_index()

# Export to files
s1_active.to_csv("../Datasets/Relative_abundance/StageOneActive_RA_long.csv", index= False)
s2_active.to_csv("../Datasets/Relative_abundance/StageTwoActive_RA_long.csv", index= False)
s1_resident_w8.to_csv("../Datasets/Relative_abundance/StageOneResidentWeekEight_RA_long.csv", index= False)
s2_resident_w8.to_csv("../Datasets/Relative_abundance/StageTwoResidentWeekEight_RA_long.csv", index= False)
s2_resident.to_csv("../Datasets/Relative_abundance/StageTwoResident_RA_long.csv", index= False)

## concatenate all subsets

In [277]:
all_data = pd.concat([s1_active, s2_active, s1_resident_w8, s2_resident_w8])

## Merge Taxonomy Data and Sample Data with Sequence Counts

In [278]:
all_data = pd.merge(left= all_data, right= sample_data, how= 'left', on= 'Sample')
all_data = pd.merge(left= all_data, right= taxonomy, how= 'left', on= 'ASV_ID')

---
# Plot Communities

### Define Function for Plotting Communities

In [455]:
# This function is only intended for this specific dataset.
# Written to save space when performing on multiple different subsets.

def plot_community(data, metadata, stage, active, level, grouping, title, outfile, barwidth= 0.8):
    # Get subset for plotting
    plot_data = data[(data['Stage'] == stage) & (data['Active'] == active)].copy()

    # Select columns and group by chosen level
    plot_data = plot_data[['Count', 'Sample', level]]
    plot_data = plot_data.groupby(['Sample', level]).sum(numeric_only= True)
    plot_data = plot_data.reset_index()

    # Rejoin with sample data
    plot_data = pd.merge(left= plot_data, right= metadata, how= 'left', on= 'Sample')

    # Only plot top 20
    top20 = plot_data.groupby(level).sum(numeric_only=1).sort_values(by= 'Count', ascending= False).index[:19]
    plot_data[level]  = plot_data[level].apply(lambda x: x if x in top20 else 'Other')
    plot_data['Count']  = plot_data['Count'].apply(lambda x: x/3)

    # create order for taxa plotting
    order = list(plot_data.groupby(level).sum(numeric_only=1).sort_values(by= 'Count', ascending= False).index)
    order.remove('Other')    # Move 'Other' to
    order.append('Other')    # the end of the list
    
    order.remove('Unassigned')    # Move 'Unassigned' to
    order.append('Unassigned')    # the end of the list

    # Plot
    plt.figure(figsize= (10,5), dpi= 200)

    figure = sns.histplot(plot_data, x= grouping, hue= level, hue_order= order, multiple= 'stack',
                 weights= 'Count', shrink= barwidth, discrete= True, palette= sns.color_palette('tab20'),
                         linewidth= 0.5, edgecolor= "Black")

    # legend & axes
    sns.move_legend(figure,loc= "upper left", bbox_to_anchor= (1,1), title= Level, fontsize= 8, ncol=1)
    plt.ylabel("Relative abundance")
    x_labels = [int(x) if isinstance(x, float) else x for x in plot_data[grouping].unique()]
    plt.xticks(ticks= plot_data[grouping].unique(), labels= x_labels)
    plt.title(title)
    plt.tight_layout()
    
    # save
    plt.savefig(f"../Figures/Community/{outfile}.png")
    plt.close()

### Plot Communities

In [456]:
# Plot communities for 

tax_levels = ["Phylum", "Class", "Order", "Family", "Genus", "Species_exact"]


for tax_level in tax_levels:
    # Stage 1 active data
    plot_community(all_data, sample_data, stage= 1, active= 1, level= tax_level, grouping= 'Temperature',
                   barwidth= 4,title = f"Stage 1 Active Communities at {tax_level} Level", 
                   outfile= f"StageOneActive_{tax_level}_community")

    # Stage 2 active data
    plot_community(all_data, sample_data, stage= 2, active= 1, level= tax_level, grouping= 'Scale of fluctuation',
                   barwidth= 0.8,title = f"Stage 2 Active Communities at {tax_level} Level",
                   outfile= f"StageTwoActive_{tax_level}_community")

    # Stage 1 resident data
    plot_community(all_data, sample_data, stage= 1, active= 0, level= tax_level, grouping= 'Temperature',
                   barwidth= 4,title = f"Stage 1 Resident Communities at {tax_level} Level",
                   outfile= f"StageOneResident_{tax_level}_community")

    # Stage 2 resident data
    plot_community(all_data, sample_data, stage= 2, active= 0, level= tax_level, grouping= 'Scale of fluctuation',
                   barwidth= 0.8,title = f"Stage 2 Resident Communities at {tax_level} Level",
                   outfile= f"StageTwoResident_{tax_level}_community")